<!-- open-in-badges -->
[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EAFIT-IA/si7011-DeepLearning/blob/main/sessions/04_representation/notebooks/sesion_04_3_masked_prediction.ipynb) [![Abrir en Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/EAFIT-IA/si7011-DeepLearning/blob/main/sessions/04_representation/notebooks/sesion_04_3_masked_prediction.ipynb) [![Abrir en Lightning Studio](https://pl-bolts-doc-images.s3.us-east-2.amazonaws.com/app-2/studio-badge.svg)](https://lightning.ai/new?repo_url=https%3A%2F%2Fgithub.com%2FEAFIT-IA%2Fsi7011-DeepLearning%2Fblob%2Fmain%2Fsessions%2F04_representation%2Fnotebooks%2Fsesion_04_3_masked_prediction.ipynb)

# SI7011 · Sesión 04 · 3. Predicción enmascarada: un MAE pequeño
**Juan David Martínez Vargas · Universidad EAFIT**

Acompaña la **Parte 3** de las diapositivas. Es para hacer **en casa**: en clase vimos la animación *Masking in motion*. Construimos un *masked autoencoder* completo, pero pequeño: imágenes de 28×28 cortadas en 16 *patches* de 7×7, un Transformer de 4 bloques como encoder y uno de 1 bloque como decoder.

| Sección | Lo que cambia | Qué mirar |
|---|---|---|
| 1 | imagen → 16 *patches* → 75 % enmascarado | qué recibe el encoder |
| 2 | el MAE y su pérdida | el encoder ve 4 tokens, el decoder 16 |
| 3 | entrenar con 75 % de máscara | qué reconstruye |
| 4 | máscara de 25 %, 50 % y 75 % | tiempo y *linear probe* |
| 5 | 100 etiquetas por clase: *probe* frente a *fine-tuning* | qué tan lineal es lo que aprendió |

**Datos:** Fashion-MNIST. Con GPU tarda unos 10 minutos; en CPU, alrededor de 40.

In [ ]:
import copy, time
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from torch import nn
from torchvision import datasets

torch.manual_seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.3})
print('PyTorch', torch.__version__, '· device', device)

train_set = datasets.FashionMNIST('data', train=True, download=True)
test_set = datasets.FashionMNIST('data', train=False, download=True)
CLASSES = train_set.classes
X_train = (train_set.data.float() / 255).unsqueeze(1).to(device)     # N × 1 × 28 × 28
X_test = (test_set.data.float() / 255).unsqueeze(1).to(device)
y_train, y_test = train_set.targets.to(device), test_set.targets.to(device)

## 1. *Patches* y máscara
Como en el ViT de la sesión 03: cada imagen se corta en una rejilla de 4×4 *patches* de 7×7 píxeles, y cada *patch* se aplana en 49 números. Para cada imagen elegimos al azar qué *patches* quedan visibles; los demás se esconden.

In [ ]:
P, N_TOK = 7, 16

def patchify(x):                                          # N × 1 × 28 × 28 → N × 16 × 49
    return x.unfold(2, P, P).unfold(3, P, P).reshape(len(x), N_TOK, P * P)

def unpatchify(t):                                        # N × 16 × 49 → N × 1 × 28 × 28
    return t.reshape(len(t), 4, 4, P, P).permute(0, 1, 3, 2, 4).reshape(len(t), 1, 28, 28)

def elegir_visibles(B, ratio, device=device):
    # por imagen: un orden aleatorio de los 16 patches; los primeros quedan visibles
    n_keep = round(N_TOK * (1 - ratio))
    orden = torch.rand(B, N_TOK, device=device).argsort(1)
    return orden[:, :n_keep], orden[:, n_keep:]          # índices visibles, índices enmascarados

assert torch.equal(unpatchify(patchify(X_test[:4])), X_test[:4])
torch.manual_seed(1)
vis, enm = elegir_visibles(6, 0.75)
t = patchify(X_test[:6]).clone()
t.scatter_(1, enm[..., None].expand(-1, -1, P * P), 0.5)  # los enmascarados, en gris
fig, axes = plt.subplots(2, 6, figsize=(8, 3))
for c in range(6):
    axes[0, c].imshow(X_test[c, 0].cpu(), cmap='gray_r'); axes[1, c].imshow(unpatchify(t)[c, 0].cpu(), cmap='gray_r', vmin=0, vmax=1)
for ax in axes.flat: ax.axis('off')
axes[1, 0].set_title('75 %: quedan 4 de 16', fontsize=8, loc='left'); plt.tight_layout(); plt.show()

**Predice:** con solo 4 *patches* visibles, ¿podrías decir qué prenda es? ¿Y dibujar el resto?

## 2. El MAE
Sigue la diapositiva *Masked autoencoders*:

1. cada *patch* visible pasa por la misma capa lineal y recibe su embedding de posición (sesión 03);
2. el **encoder** $g$, de 4 bloques, procesa **solo los tokens visibles**;
3. el **decoder** $d$, de 1 bloque, recibe los 16 lugares: los tokens codificados en su posición y un token aprendido **[M]** en cada lugar enmascarado, más la posición;
4. una capa lineal predice los 49 píxeles de cada *patch*, y la pérdida cuenta **solo los enmascarados**:
$$L_i = \frac{1}{|M|} \sum_{m \in M} \lVert \mathbf x_m - \hat{\mathbf x}_m \rVert^2$$

Para la *probe*, el encoder ve la imagen **completa** (los 16 *patches*) y $\mathbf h$ es el promedio de sus 16 tokens: 64 números.

In [ ]:
def bloque(d):
    return nn.TransformerEncoderLayer(d, nhead=4, dim_feedforward=2 * d, dropout=0.0, batch_first=True, norm_first=True)

class MAE(nn.Module):
    def __init__(self, d=64, depth=4, dec_depth=1):
        super().__init__()
        self.embed = nn.Linear(P * P, d)                                     # el mismo W_E para cada patch
        self.pos = nn.Parameter(torch.randn(1, N_TOK, d) * 0.02)             # posición del encoder
        self.enc = nn.Sequential(*[bloque(d) for _ in range(depth)], nn.LayerNorm(d))        # g
        self.mask_token = nn.Parameter(torch.zeros(1, 1, d))                 # [M], aprendido
        self.dec_pos = nn.Parameter(torch.randn(1, N_TOK, d) * 0.02)         # posición del decoder
        self.dec = nn.Sequential(*[bloque(d) for _ in range(dec_depth)], nn.LayerNorm(d))    # d
        self.head = nn.Linear(d, P * P)                                      # token → 49 píxeles

    def encode(self, x, vis=None):
        t = self.embed(patchify(x)) + self.pos
        if vis is not None:                                                  # solo los visibles entran a g
            t = torch.gather(t, 1, vis[..., None].expand(-1, -1, t.shape[-1]))
        return self.enc(t)

    def forward(self, x, ratio):
        vis, enm = elegir_visibles(len(x), ratio, x.device)
        h = self.encode(x, vis)                                              # B × n_visibles × d
        lugares = self.mask_token.expand(len(x), N_TOK, -1).clone()          # 16 lugares, todos [M] ...
        lugares.scatter_(1, vis[..., None].expand(-1, -1, h.shape[-1]), h)   # ... y los visibles en su posición
        pred = self.head(self.dec(lugares + self.dec_pos))                   # B × 16 × 49
        err = ((pred - patchify(x)) ** 2).mean(-1)                           # error por patch
        M = torch.zeros_like(err).scatter_(1, enm, 1.0)
        return (err * M).sum() / M.sum(), pred, enm                          # L_i solo sobre los enmascarados

    @torch.no_grad()
    def h(self, X, chunk=2000):                                              # h = promedio de los 16 tokens de g(x)
        self.eval(); H = torch.cat([self.encode(X[i:i + chunk]).mean(1) for i in range(0, len(X), chunk)]); self.train()
        return H

m = MAE().to(device)
print(f'parámetros: encoder {sum(p.numel() for p in m.enc.parameters()):,} · decoder {sum(p.numel() for p in m.dec.parameters()):,}')

In [ ]:
def entrenar(ratio, pasos=3000, lr=1e-3, batch=256):
    torch.manual_seed(0)
    m = MAE().to(device)
    opt = torch.optim.AdamW(m.parameters(), lr=lr, weight_decay=0.05)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, lr, total_steps=pasos, pct_start=0.1)   # warmup + bajada
    paso, t0 = 0, time.time()
    while paso < pasos:
        for b in torch.randperm(len(X_train), device=device).split(batch):
            loss, _, _ = m(X_train[b], ratio)                 # sin etiquetas
            opt.zero_grad(); loss.backward(); opt.step(); sched.step()
            paso += 1
            if paso == pasos: break
    dt = time.time() - t0
    print(f'máscara {ratio:.0%} · pérdida final {loss.item():.4f} · {dt:.0f} s')
    return m, dt

def probe(H_train, H_test, ytr=None, steps=500):
    ytr = y_train if ytr is None else ytr
    mu, sd = H_train.mean(0), H_train.std(0) + 1e-6
    torch.manual_seed(0)
    head = nn.Linear(H_train.shape[1], 10).to(device)
    opt = torch.optim.Adam(head.parameters(), lr=1e-2)
    for _ in range(steps):
        loss = F.cross_entropy(head((H_train - mu) / sd), ytr); opt.zero_grad(); loss.backward(); opt.step()
    with torch.no_grad():
        return (head((H_test - mu) / sd).argmax(1) == y_test).float().mean().item()

## 3. Entrenar con 75 % de máscara

In [ ]:
modelos, tiempos = {}, {}
modelos[0.75], tiempos[0.75] = entrenar(0.75)

torch.manual_seed(3)
x = X_test[:8]
with torch.no_grad():
    modelos[0.75].eval(); _, pred, enm = modelos[0.75](x, 0.75); modelos[0.75].train()
entrada = patchify(x).clone().scatter_(1, enm[..., None].expand(-1, -1, P * P), 0.5)
salida = patchify(x).clone().scatter_(1, enm[..., None].expand(-1, -1, P * P), torch.gather(pred, 1, enm[..., None].expand(-1, -1, P * P)))
fig, axes = plt.subplots(3, 8, figsize=(9, 3.6))
for c in range(8):
    for r, im in enumerate([unpatchify(entrada), unpatchify(salida), x]):
        axes[r, c].imshow(im[c, 0].cpu().clamp(0, 1), cmap='gray_r', vmin=0, vmax=1); axes[r, c].axis('off')
for r, n in enumerate(['lo que ve g', 'visibles + predichos', 'original']):
    axes[r, 0].set_title(n, fontsize=8, loc='left')
plt.tight_layout(); plt.show()

Las predicciones son borrosas, como en la diapositiva: la red predice el **promedio** de lo que podría haber detrás de la máscara, y el promedio de muchas prendas posibles es una mancha. Pero la forma general (manga larga o corta, pantalón, zapato) suele estar bien, y eso solo se puede adivinar entendiendo cómo se relacionan las partes de una prenda.

## 4. ¿Cuánto enmascarar?
**Predice** antes de correr: ¿qué razón de máscara dará la mejor *probe* sobre $\mathbf h$? ¿Cuál entrenará más rápido?

In [ ]:
for r in [0.25, 0.50]:
    modelos[r], tiempos[r] = entrenar(r)
torch.manual_seed(0)
azar = MAE().to(device)
res = {'aleatorio': {'tiempo_s': 0.0, 'probe': probe(azar.h(X_train), azar.h(X_test))}}
for r in [0.25, 0.50, 0.75]:
    res[f'{r:.0%}'] = {'tiempo_s': tiempos[r], 'probe': probe(modelos[r].h(X_train), modelos[r].h(X_test))}
{n: {k: round(v, 3) for k, v in d.items()} for n, d in res.items()}

Compara las dos columnas. El **tiempo** baja al subir la máscara, siempre: el encoder procesa 12, 8 o 4 tokens, y es la parte cara (diapositiva *Why mask so much?*). La ***probe***, en cambio, cambia poco entre razones, uno o dos puntos, y en Fashion-MNIST no suele ganar el 75 %. Las tres le ganan con claridad al Transformer aleatorio.

¿Por qué no se repite el 75 % del artículo? En ImageNet una imagen de 224×224 tiene 196 *patches* de 16×16, y un *patch* se parece mucho a sus vecinos: con 25 % de máscara la tarea se resuelve copiando de al lado. Aquí cada *patch* es la sexta parte de una manga, hay poca redundancia, y esconder 12 de 16 ya deja muy poco para adivinar. **La razón de máscara depende de cuán redundantes son los datos**, que es justo el argumento de la diapositiva.

## 5. Con 100 etiquetas por clase: *probe* frente a *fine-tuning*
La diapositiva *What masked prediction learns* dice que las features de un MAE rinden poco con una *probe* lineal y mucho mejor después de *fine-tuning*: la información está en $\mathbf h$, pero no ordenada para que una sola capa lineal la lea. Lo probamos con 1 000 imágenes etiquetadas, comparando el encoder del MAE con el mismo Transformer desde pesos aleatorios.

In [ ]:
gen = torch.Generator().manual_seed(0)
i100 = torch.cat([(y_train == c).nonzero().squeeze(1)[torch.randperm(6000, generator=gen)[:100].to(device)] for c in range(10)])

def finetune(m0, epocas=30):
    torch.manual_seed(0)
    m = copy.deepcopy(m0); head = nn.Linear(64, 10).to(device)
    opt = torch.optim.AdamW([*m.parameters(), *head.parameters()], lr=5e-4, weight_decay=0.05)
    for _ in range(epocas):
        for b in i100[torch.randperm(len(i100), device=device)].split(64):
            loss = F.cross_entropy(head(m.encode(X_train[b]).mean(1)), y_train[b])
            opt.zero_grad(); loss.backward(); opt.step()
    m.eval()
    with torch.no_grad():
        return (torch.cat([head(m.encode(X_test[i:i + 2000]).mean(1)) for i in range(0, len(X_test), 2000)]).argmax(1)
                == y_test).float().mean().item()

pocas = {}
for n, mod in [('aleatorio', azar), ('MAE 75 %', modelos[0.75])]:
    H_tr, H_te = mod.h(X_train), mod.h(X_test)
    pocas[n] = {'probe': probe(H_tr[i100], H_te, ytr=y_train[i100]), 'fine-tuning': finetune(mod)}
{n: {k: round(v, 3) for k, v in d.items()} for n, d in pocas.items()}

El preentrenamiento ayuda mucho a la *probe*: con las mismas 1 000 etiquetas, $\mathbf h$ del MAE le gana por unos 10 puntos al Transformer aleatorio. Al *fine-tuning* le ayuda bastante menos, porque en Fashion-MNIST mil etiquetas ya alcanzan para entrenar un Transformer pequeño casi desde cero.

En los dos encoders el *fine-tuning* supera a la *probe*, pero en el MAE la distancia es pequeña. El efecto de la diapositiva aparece con modelos y datos grandes: en el artículo del MAE, con un ViT-L en ImageNet, la *probe* queda unos 10 puntos por debajo del *fine-tuning*. Compáralo con SimCLR en el notebook 5, donde la *probe* sola ya rinde casi tanto como ajustar todo.

## Tu turno
1. Entrena con máscara del 90 % (quedan 2 *patches*). ¿Hasta dónde se puede subir antes de que la *probe* caiga?
2. Cambia el tamaño de los *patches* a 4×4 (`P = 4`, 49 tokens). Con más tokens, ¿cambia la razón de máscara que más conviene? (pista: diapositiva *Why mask so much?*).
3. Calcula la pérdida también sobre los *patches* visibles. ¿Mejora o empeora la *probe*? ¿Por qué el MAE no lo hace?

**Para pensar.** BERT (sesión 05) enmascara solo el 15 % de las palabras, y el MAE de imágenes el 75 % de los *patches*. ¿Qué diferencia entre el texto y las imágenes explica esa diferencia?

In [ ]:
# Experimento 1: máscara del 90 %

### Referencias
- He, Chen, Xie, Li, Dollár y Girshick (2022), *Masked Autoencoders Are Scalable Vision Learners*, CVPR.
- Devlin, Chang, Lee y Toutanova (2019), *BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding*, NAACL.
- Pathak et al. (2016), *Context Encoders: Feature Learning by Inpainting*, CVPR (la idea antes de los Transformers).